[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# The Estimator API


## What you will be able to do

Make a scikit-learn model, train it with `fit`, ask it for answers with `predict`, and read what it
learned, knowing that every other model in the library takes the same three steps.

## The idea

### The problem

A weather network has 1,200 stations, and for each one you know where it stands: its latitude, its
height above the sea, how far it is from the coast. You also know each station's mean temperature
in January. A new station is planned on a hillside in the north, and the question is what January
will be like there before anyone has measured it.

The **NumPy, Deep Dive** guide could fit a straight line to numbers like these with least squares,
and the **Pandas, Deep Dive** guide could hold the table. What neither gives you is a choice of
models behind one way of working. A straight line is one guess about how temperature depends on
place. A tree of yes-or-no questions is another, and a forest of such trees a third. Written from
scratch, each would need its own code for training, for answering and for scoring, and comparing
them would mean learning three interfaces.

scikit-learn solves that by giving every model the same small set of methods. Once you can use one,
you can use all of them, and swapping one model for another changes a single line.

### What an estimator is

> In scikit-learn, an **estimator** is an object that learns from data. You make one by calling its
> class with its settings, and its **`fit(X, y)`** method learns from a table of features `X`, one
> row per example and one column per feature, and from `y`, the answer for each row. A
> **predictor** is an estimator that also has **`predict(X)`**, which answers for new rows. What
> `fit` learned is stored on the object in attributes whose names end in an underscore, such as
> `coef_`.

### Why it works that way

Separating the settings from the learning is what makes the models interchangeable. The settings go
into the class when you make the object, such as how deep a tree may grow, and they never depend on
the data. The data goes into `fit`, and nothing about the data is fixed until then. So the same
object, unfitted, can be copied, given other settings, or fitted again on other rows, and code that
calls `fit` and `predict` never needs to know which model it holds.

The trailing underscore is a promise about time. A name without one, such as `max_depth`, is a
setting you chose. A name with one, such as `coef_`, exists only after `fit`, and asking a model for
it, or for a prediction, before then raises an error that says so.

`X` is always two-dimensional, even with a single feature, because a model sees rows of features.
`y` is one-dimensional, an answer per row. A pandas DataFrame and Series fit those shapes, and when
`X` is a DataFrame the model remembers its column names and checks them when you predict.

### Where this shows up

Every model in scikit-learn follows this, from linear models to forests and nearest neighbors, and
so do the tools that prepare data, which have `fit` and `transform` in place of `predict`. The rest
of this guide builds on it. The **Pipelines** notebook chains estimators, and the **Hyperparameter
Search** notebook tries many settings of one. The statistics behind each model are the subject of
the Visual Handbook, and this guide covers the library.

### What this notebook covers

- the stations as a table of features and a column of answers
- making, fitting and asking a linear model, and reading `coef_`
- a decision tree and a random forest, fitted with the same three lines
- settings with `get_params` and `set_params`, and `fit` returning the model
- a classifier with `predict_proba`, and a scaler with `transform`
- four errors, all of them about what a model expects to be given

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
from sklearn.linear_model import LinearRegression

X = [[1.0], [2.0], [3.0], [4.0]]
y = [3.1, 4.9, 7.2, 8.8]

model = LinearRegression()
model.fit(X, y)
print(model.predict([[5.0]]).round(2))
print(model.coef_.round(2), round(model.intercept_, 2))
```

```
[10.85]
[1.94] 1.15
```

`LinearRegression()` made a model with its default settings, `fit` found the line through the four
points that fits them best, and `predict` used that line for a fifth point. The slope, 1.94, and the
intercept, 1.15, are what `fit` learned, and both names end in an underscore.

## Setup

Two libraries for the data, scikit-learn, and four of its models and tools.

- `numpy` and `pandas` build the stations, and the cell prints scikit-learn's version
- `LinearRegression`, `DecisionTreeRegressor` and `RandomForestRegressor` are three models of January
  temperature, used side by side in the worked examples
- `DecisionTreeClassifier` answers a yes-or-no question, whether a station has snow on 1 May
- `StandardScaler` is a transformer, which prepares data instead of predicting

`station_network` makes a network of 1,200 weather stations in Norway. The stations are made up, and
a generator started from a fixed seed draws them, so the table is the same on every run. Each row
has a station's latitude, longitude, elevation in meters, distance from the coast in kilometers and
the year it was installed, its terrain and owner as text, and three things to predict: its January
mean in degrees Celsius, whether it has snow on 1 May, and whether it has a frost in June.
`FEATURES` names the five numeric columns the models use here.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


## Worked examples

### The stations as features and answers

In [2]:
network.head()

,station,latitude,longitude,elevation_m,coast_km,terrain,owner,installed,jan_mean,snow_on_may_1,june_frost
0,ST0001,70.93,29.07,358,17.0,inland,met,1987,-9.6,True,False
1,ST0002,62.97,16.58,311,11.1,fjord,met,2021,-3.7,False,False
2,ST0003,68.75,21.61,84,10.0,inland,airport,2009,-4.8,False,False
3,ST0004,68.88,22.50,239,60.0,inland,private,1994,-9.4,True,False
4,ST0005,70.69,26.28,461,19.3,inland,met,1970,-8.2,True,False


Each row is one station. For a model, the five columns in `FEATURES` are the **features**, what is
known about a station before anyone measures its January, and `jan_mean` is the **target**, the
answer to learn. `X` and `y` are the names scikit-learn's documentation uses for them:

In [3]:
X = network[FEATURES]
y = network["jan_mean"]

print(type(X).__name__, X.shape)
print(type(y).__name__, y.shape)

DataFrame (1200, 5)
Series (1200,)


`X` is a DataFrame of 1,200 rows and five columns, and `y` a Series of 1,200 answers, one for each
row of `X`.

### Make, fit and predict

A model is made in one line, with its settings, and learns in another:

In [4]:
linear = LinearRegression()
linear.fit(X, y)

print("predicted:", linear.predict(X.head(3)).round(1))
print("measured: ", y.head(3).to_numpy())

predicted: [-8.8 -3.7 -5.4]
measured:  [-9.6 -3.7 -4.8]


`predict` takes rows in the same shape as `X` and returns one answer per row, as a NumPy array. These
three rows were among the 1,200 the model learned from, so the predictions say only that the call
works. Whether a model is any good is a question about rows it has not seen, which the **Your First
Model** notebook asks.

### What fit learned

A linear model learned one coefficient for each feature, and an intercept:

In [5]:
print(pd.Series(linear.coef_, index=FEATURES).round(4))
print("intercept:", round(linear.intercept_, 2))

latitude      -0.5224
longitude     -0.0144
elevation_m   -0.0063
coast_km      -0.0444
installed      0.0037
dtype: float64
intercept: 24.44


Each coefficient is the change in the predicted January mean for one unit more of its feature, with
the others held still. A degree of latitude further north makes the prediction 0.52 degrees colder,
and a hundred meters of height 0.63 degrees colder. `coef_` and `intercept_` did not exist before
`fit`, which is what their underscores mean, and the model kept the column names too:

In [6]:
print(linear.feature_names_in_)
print(linear.n_features_in_)

['latitude' 'longitude' 'elevation_m' 'coast_km' 'installed']
5


### Another model with the same three lines

A decision tree asks a series of yes-or-no questions about the features, and a random forest averages
many trees. Their settings differ, and the lines that train and use them do not:

In [7]:
models = {
    "linear": LinearRegression(),
    "tree": DecisionTreeRegressor(max_depth=4, random_state=0),
    "forest": RandomForestRegressor(random_state=0),
}

for name, model in models.items():
    model.fit(X, y)
    print(f"{name:7} {model.predict(X.head(3)).round(1)}")

linear  [-8.8 -3.7 -5.4]
tree    [-8.2 -4.  -6.4]
forest  [-9.3 -3.7 -5.1]


The loop did not know which model it held. `random_state=0` fixes the random choices the tree and the
forest make while they learn, so they give the same answers on every run. What they learned is
stored differently, though, since a tree has no coefficients. Its splits are in `tree_`, and both
tree models report how much each feature was used:

In [8]:
print(pd.Series(models["forest"].feature_importances_, index=FEATURES).round(3))

latitude       0.480
longitude      0.044
elevation_m    0.209
coast_km       0.249
installed      0.017
dtype: float64


Latitude carries the most, close to half. The **Feature Importance** notebook explains what these numbers
measure and where they mislead.

### Settings are part of the object

`get_params` lists every setting, with the defaults filled in, and `set_params` changes them:

In [9]:
tree = DecisionTreeRegressor(max_depth=4)
print(tree.get_params()["max_depth"], tree.get_params()["min_samples_leaf"])

tree.set_params(max_depth=2)
print(tree.get_params()["max_depth"])

4 1
2


Changing a setting changes nothing learned, because nothing has been learned yet. `fit` also returns
the model itself, so a model can be made and trained in one expression:

In [10]:
deep = DecisionTreeRegressor(max_depth=8, random_state=0).fit(X, y)

print(type(deep).__name__, deep.get_depth())

DecisionTreeRegressor 8


### A classifier has the same shape

A **classifier** predicts a category instead of a number. Here the question is whether a station has
snow on the ground on 1 May:

In [11]:
snow = network["snow_on_may_1"]
classifier = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, snow)

print("classes:  ", classifier.classes_)
print("predicted:", classifier.predict(X.head(3)))
print("measured: ", snow.head(3).to_numpy())
print(classifier.predict_proba(X.head(3)).round(2))

classes:   [False  True]
predicted: [ True False False]
measured:  [ True False False]
[[0.19 0.81]
 [0.97 0.03]
 [0.63 0.37]]


`classes_` lists the answers it can give, in order, `False` and then `True`. `predict_proba` gives a
probability for each, in the same order, so the first station's 0.81 in the second column is the
chance it puts on snow. `predict` chooses the more likely class, and the **Classification Metrics**
notebook shows when to choose differently.

### A transformer has the same shape too

A tool that prepares data is an estimator as well. `StandardScaler` learns each column's mean and
spread in `fit`, and `transform` uses them to put every column on the same scale:

In [12]:
scaler = StandardScaler().fit(X)

print(scaler.mean_.round(2))
print(scaler.transform(X.head(2)).round(2))

[  64.45   17.28  292.8    24.63 1987.53]
[[ 1.74  2.06  0.34 -0.32 -0.02]
 [-0.4  -0.12  0.1  -0.56  1.57]]


`mean_` and `scale_`, the learned mean and standard deviation, end in an underscore like `coef_`.
After the transform, each value is the number of standard deviations it lies from its column's mean,
so an elevation and a year can sit side by side. The **Preprocessing** notebook covers why a model
needs that, and why the scaler must learn from the training rows alone.

### What score measures on the training rows

Every predictor has `score(X, y)`. For a regressor it is R², the share of the target's variation
the predictions account for, where 1.0 is perfect:

In [13]:
for name, model in models.items():
    print(f"{name:7} {model.score(X, y):.3f}")

linear  0.882
tree    0.801
forest  0.984


The forest seems far ahead, at 0.984. These scores are on the rows each model learned from, which is
the one place a model can look better than it is, and the **Your First Model** notebook scores the
same models on rows they have not seen.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/01-the-estimator-api-solutions.ipynb).

**1.** Fit a `LinearRegression` that predicts `jan_mean` from `latitude` and `elevation_m` alone, and print its two coefficients with their feature names.

In [14]:
# your code here

**2.** Predict the January mean of a planned station at latitude 69.5, longitude 23.0, elevation 450 m, 21 km from the coast, installed in 2026, with the `linear` model. Build the row as a one-row DataFrame with the columns in `FEATURES`.

In [15]:
# your code here

**3.** Make a `DecisionTreeRegressor` with `max_depth=2` and `random_state=0`, fit it, and print its `get_depth()` and its `score` on `X` and `y`.

In [16]:
# your code here

**4.** Fit a `DecisionTreeClassifier(max_depth=3, random_state=0)` on `june_frost` instead of snow, and print `classes_` and how many of the 1,200 stations it predicts will have a frost in June.

In [17]:
# your code here

**5.** Use `StandardScaler` on `X`, and print the mean and standard deviation of every transformed column, rounded to two places.

In [18]:
# your code here

## Common errors

### NotFittedError: This LinearRegression instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [19]:
LinearRegression().predict(X.head(3))

NotFittedError: This LinearRegression instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

A new model has its settings and nothing learned, so it has nothing to predict with. Call `fit` first.
The same error comes from asking an unfitted model for `coef_` or any other name with an underscore.

### ValueError: Expected a 2-dimensional container but got <class 'pandas.core.series.Series'> instead.

In [20]:
LinearRegression().fit(network["latitude"], y)

ValueError: Expected a 2-dimensional container but got <class 'pandas.core.series.Series'> instead. Pass a DataFrame containing a single row (i.e. single sample) or a single column (i.e. single feature) instead.

`network["latitude"]` is a Series, one-dimensional, and `X` has to be a table even when it holds a
single feature. Double brackets make a one-column DataFrame, `network[["latitude"]]`, and the rest of
the message says the same.

### ValueError: The feature names should match those that were passed during fit.

In [21]:
linear.predict(X[["latitude", "longitude", "elevation_m", "coast_km"]].head(3))

ValueError: The feature names should match those that were passed during fit.
Feature names seen at fit time, yet now missing:
- installed


`linear` learned from five named columns, and these rows have four. The message goes on to name the
one that is missing, `installed`. A model needs every feature it was trained on, under the same
names and in the same order, and columns in another order raise the same error.

### UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names

In [22]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    answers = linear.predict(X.head(3).to_numpy())

print(answers.round(1))
for warning in caught:
    print(f"{warning.category.__name__}: {warning.message}")

[-8.8 -3.7 -5.4]


A NumPy array has no column names, so the model cannot check that the columns are the ones it learned
from, in the order it learned them. It predicts by position, and warns. The answers are right here
only because the array kept `FEATURES`' order. `warnings` is imported in this cell to catch the
warning and print it on one line. Give a model a DataFrame with the same columns, and the check runs.

## Recap

- Make a model with its settings, train it with `fit(X, y)`, and answer with `predict(X)`
- Read what a model learned from the attributes that end in an underscore, such as `coef_`
- Swap one model for another without changing the code that fits and uses it
- Get a classifier's probabilities with `predict_proba`, and transform data with a scaler
- Give a model a two-dimensional `X` with the columns it was trained on

## What is next

The **Your First Model** notebook holds some stations back, trains on the rest, and scores the models
on stations they have not seen.

---

[scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
